In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND FACT CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
SILVER_RETEST = f"{CATALOG}.silver.simulated_retest_events"
DIM_DATE = f"{CATALOG}.gold.dim_date"
DIM_LOT = f"{CATALOG}.gold.dim_lot"
FACT_LOT = f"{CATALOG}.gold.fact_lot_performance"
FACT_PERIODIC = f"{CATALOG}.gold.fact_yield_periodic"
FACT_RETEST = f"{CATALOG}.gold.fact_retest_equipment"

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Day 3 retest fact configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — BUILD THE RETEST EQUIPMENT FACT (PYTHON)
# ===================================================

retest_source_df = spark.table(SILVER_RETEST)
dim_lot_df = spark.table(DIM_LOT)

retest_fact_df = (
    retest_source_df.alias("source")
    .join(
        dim_lot_df.alias("lot"),
        F.col("source.source_lot_id") == F.col("lot.source_lot_id"),
        "left",
    )
    .join(
        spark.table(DIM_DATE).alias("date"),
        F.col("lot.production_date") == F.col("date.full_date"),
        "left",
    )
    .withColumn("retest_hour_utc", F.date_trunc("hour", F.col("retest_timestamp_utc")))
    .groupBy(
        F.col("retest_hour_utc"),
        F.col("lot.production_date"),
        F.col("date.date_key"),
        F.col("lot.lot_key"),
        F.col("lot.site_key"),
        F.col("lot.product_group_key"),
        F.col("lot.device_key"),
        F.col("lot.equipment_key"),
        F.col("source.source_lot_id"),
        F.col("source.defect_code"),
        F.col("source.error_code"),
        F.col("source.simulation_seed"),
        F.col("source.simulation_version"),
        F.col("source.simulated_record_flag"),
        F.col("source.record_origin"),
    )
    .agg(
        F.countDistinct("retest_event_id").alias("retest_event_count"),
        F.sum("retest_input_quantity").alias("retest_input_quantity"),
        F.sum("retest_good_quantity").alias("retest_good_quantity"),
        F.sum("retest_fail_quantity").alias("retest_fail_quantity"),
        F.avg("retest_test_time_seconds").alias("average_retest_test_time_seconds"),
        F.max("retest_timestamp_utc").alias("latest_retest_timestamp_utc"),
    )
    .withColumn(
        "retest_pass_rate",
        F.when(
            F.col("retest_input_quantity") > 0,
            F.col("retest_good_quantity").cast("double")
            / F.col("retest_input_quantity").cast("double"),
        ),
    )
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

unresolved_keys = retest_fact_df.filter(
    F.col("date_key").isNull()
    | F.col("lot_key").isNull()
    | F.col("site_key").isNull()
    | F.col("product_group_key").isNull()
    | F.col("device_key").isNull()
    | F.col("equipment_key").isNull()
).count()
assert unresolved_keys == 0, f"Unresolved retest fact keys: {unresolved_keys}"

(
    retest_fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_RETEST)
)

print(f"Retest-equipment fact rows: {spark.table(FACT_RETEST).count():,}")

In [0]:
# ===================================================
# BLOCK 3 — UPDATE THE LOT PERFORMANCE FACT (PYTHON)
# ===================================================

retest_by_lot_df = (
    spark.table(SILVER_RETEST)
    .groupBy("source_lot_id")
    .agg(
        F.sum("retest_input_quantity").alias("new_retest_input_quantity"),
        F.sum("retest_good_quantity").alias("new_retest_good_quantity"),
        F.sum("retest_fail_quantity").alias("new_retest_fail_quantity"),
    )
)

updated_lot_fact_df = (
    spark.table(FACT_LOT).alias("fact")
    .join(retest_by_lot_df.alias("retest"), "source_lot_id", "left")
    .select(
        "fact.*",
        F.col("retest.new_retest_input_quantity"),
        F.col("retest.new_retest_good_quantity"),
        F.col("retest.new_retest_fail_quantity"),
    )
    .withColumn(
        "retest_input_quantity",
        F.coalesce(F.col("new_retest_input_quantity"), F.lit(0)).cast("long"),
    )
    .withColumn(
        "retest_good_quantity",
        F.coalesce(F.col("new_retest_good_quantity"), F.lit(0)).cast("long"),
    )
    .withColumn(
        "retest_fail_quantity",
        F.coalesce(F.col("new_retest_fail_quantity"), F.lit(0)).cast("long"),
    )
    .withColumn(
        "final_good_quantity",
        F.col("first_pass_good_quantity") + F.col("retest_good_quantity"),
    )
    .withColumn(
        "final_test_yield",
        F.when(
            F.col("input_quantity") > 0,
            F.col("final_good_quantity").cast("double")
            / F.col("input_quantity").cast("double"),
        ),
    )
    .withColumn(
        "retest_recovery_contribution",
        F.col("final_test_yield") - F.col("first_pass_yield"),
    )
    .withColumn("retest_data_available_flag", F.lit(True))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
    .drop(
        "new_retest_input_quantity",
        "new_retest_good_quantity",
        "new_retest_fail_quantity",
    )
    .localCheckpoint(eager=True)
)

assert updated_lot_fact_df.count() == spark.table(DIM_LOT).count()

(
    updated_lot_fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_LOT)
)

print(f"Updated lot fact rows: {spark.table(FACT_LOT).count():,}")

In [0]:
# ===================================================
# BLOCK 4 — REBUILD THE PERIODIC YIELD FACT (PYTHON)
# ===================================================

periodic_df = (
    spark.table(FACT_LOT)
    .groupBy(
        "date_key",
        "production_date",
        "site_key",
        "product_group_key",
        "device_key",
    )
    .agg(
        F.count("*").alias("lot_count"),
        F.sum(F.col("tested_lot_flag").cast("long")).alias("tested_lot_count"),
        F.sum("input_quantity").alias("input_quantity"),
        F.sum("first_pass_good_quantity").alias("first_pass_good_quantity"),
        F.sum("first_pass_fail_quantity").alias("first_pass_fail_quantity"),
        F.sum(
            F.when(F.col("first_pass_status") == "FPY_BELOW_TARGET", 1).otherwise(0)
        ).alias("first_pass_below_target_lot_count"),
        F.sum("retest_input_quantity").alias("retest_input_quantity"),
        F.sum("retest_good_quantity").alias("retest_good_quantity"),
        F.sum("retest_fail_quantity").alias("retest_fail_quantity"),
        F.sum("final_good_quantity").alias("final_good_quantity"),
    )
    .withColumn(
        "first_pass_yield",
        F.when(
            F.col("input_quantity") > 0,
            F.col("first_pass_good_quantity") / F.col("input_quantity"),
        ),
    )
    .withColumn(
        "final_test_yield",
        F.when(
            F.col("input_quantity") > 0,
            F.col("final_good_quantity") / F.col("input_quantity"),
        ),
    )
    .withColumn(
        "retest_recovery_rate",
        F.when(
            F.col("retest_input_quantity") > 0,
            F.col("retest_good_quantity") / F.col("retest_input_quantity"),
        ),
    )
    .withColumn(
        "retest_recovery_contribution",
        F.col("final_test_yield") - F.col("first_pass_yield"),
    )
    .withColumn("retest_data_available_flag", F.lit(True))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

(
    periodic_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_PERIODIC)
)

print(f"Updated periodic fact rows: {spark.table(FACT_PERIODIC).count():,}")


In [0]:
# ===================================================
# BLOCK 5 — FACT BUILD SUMMARY (PYTHON)
# ===================================================

summary = [
    ("fact_retest_equipment", spark.table(FACT_RETEST).count()),
    ("fact_lot_performance", spark.table(FACT_LOT).count()),
    ("fact_yield_periodic", spark.table(FACT_PERIODIC).count()),
]
display(spark.createDataFrame(summary, ["table_name", "row_count"]))